# Benchmark & optimize a vision model on real edge devices — TinyEdge

Two calls against [TinyEdge](https://tinyedge.ai)'s device cloud:

1. **Benchmark** an ONNX classifier on real phones/tablets → measured latency, throughput, and **top-1 accuracy on your own test set, computed on-device**.
2. The same call with **`optimize=True`** → TinyEdge builds a calibrated **int8** variant (with a local safety gate that refuses architectures int8 breaks), benchmarks original + variant as one sweep, and shows the measured speed/accuracy trade. On TinyEdge's reference fleet this is **~3.4× faster at −0.6 points**.

**Before you run:** a [tinyedge.ai](https://tinyedge.ai) account ($25 demo credit; this notebook uses ~$0.50) · paste your API key in the first code cell · at least one device paired via the TinyEdge Runner app with *"Available for benchmarks"* on · Kaggle **Internet enabled** (right sidebar; needs a phone-verified Kaggle account).

In [ ]:
%pip -q install "tinyedge>=0.2.4" onnxruntime

In [ ]:
import tinyedge

# Paste your key from tinyedge.ai → New benchmark → "Your API key":
client = tinyedge.TinyEdge(api_key="tinyedge_sk_REPLACE_ME")

DEVICES = client.devices(online=True)
print("benchmarking on:", DEVICES)

**The model** — pretrained MobileNetV2, exported to ONNX right here (PyTorch never leaves this notebook; devices only ever see the ONNX).

In [ ]:
import torch
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

model = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1).eval()
torch.onnx.export(model, torch.randn(1, 3, 224, 224), "mobilenet_v2.onnx",
                  input_names=["input"], output_names=["output"],
                  dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
                  dynamo=False)
print("mobilenet_v2.onnx exported")

**The test set** — [Imagenette](https://github.com/fastai/imagenette) (10 easy ImageNet classes, ~100 MB). TinyEdge's convention: *numeric folder names are the model's output class indices*, so we map each Imagenette class folder to its ImageNet index. 30 images per class keeps the on-device run quick; the same images double as int8 calibration data.

In [ ]:
import pathlib, shutil, tarfile, urllib.request

if not pathlib.Path("imagenette2-160").exists():
    urllib.request.urlretrieve(
        "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz", "imagenette.tgz")
    tarfile.open("imagenette.tgz").extractall(".")

IMAGENET_IDX = {"n01440764": 0, "n02102040": 217, "n02979186": 482, "n03000684": 491,
                "n03028079": 497, "n03394916": 566, "n03417042": 569, "n03425413": 571,
                "n03445777": 574, "n03888257": 701}

testset = pathlib.Path("testset")
for wnid, idx in IMAGENET_IDX.items():
    dst = testset / str(idx); dst.mkdir(parents=True, exist_ok=True)
    for f in sorted((pathlib.Path("imagenette2-160/val") / wnid).iterdir())[:30]:
        shutil.copy(f, dst / f.name)
print("testset: 10 classes x 30 images")

## Benchmark — the model, as-is

In [ ]:
# one result per device — latency, throughput, on-device top-1 accuracy
client.benchmark("mobilenet_v2.onnx", devices=DEVICES, dataset="testset")

## Optimize — same call, one extra keyword
TinyEdge quantizes to int8 (calibrated on *your* images), checks locally that int8 didn't break *your* model, then benchmarks original + int8 on every device as one sweep.

In [ ]:
report = client.benchmark("mobilenet_v2.onnx", devices=DEVICES,
                          dataset="testset", optimize=True)
print(report.summary())
print("full report with charts + AI analysis:", report.sweep_url)

## Reading the result

Expect int8 to be **several times faster at nearly identical accuracy** for MobileNetV2 — it's a quantization-friendly architecture. Try the same two calls with your own model: if it uses squeeze-excite blocks or hard-swish (MobileNetV3-Small, EfficientNet…), the safety gate will *exclude* the int8 variant and tell you why — that's TinyEdge measuring instead of assuming.

More: [docs](https://tinyedge.ai/docs) · [public measured benchmark data](https://huggingface.co/datasets/TinyEdge/edge-inference-benchmarks) · LLMs work the same way — see the companion notebook `tinyedge_llm_optimize_kaggle.ipynb`.

## What should you ship? (the decision layer)

The numbers above are evidence — this is the **decision**. TinyEdge computes, server-side from your sweep, the **failure threshold** (the quant where quality collapses), a **compression-efficiency score**, and the **best config per deployment goal** (max quality / cheapest viable / real-time / battery).

In [ ]:
# Deployment decision — computed server-side from the sweep above.
import requests

gid = report.sweep_url.rstrip("/").split("/")[-1]
intel = requests.get(
    f"{client.base}/benchmarks/sweep/{gid}",
    headers={"Authorization": f"Bearer {client.api_key}"}, timeout=60,
).json().get("intelligence") or {}

curves = intel.get("curves")
if not curves:
    print("Decision layer needs >=2 quants on a shared quality axis - re-run the optimize cell.")
else:
    st = curves.get("stability") or {}
    print("Compression efficiency (CES):", curves.get("ces"), " (0-1, higher = quality kept)")
    print("Degradation stability:       ", st.get("score"),
          "(non-monotonic ladder)" if st.get("nonMonotonic") else "")
    ft = curves.get("failureThreshold")
    print()
    if ft:
        print("FAILURE THRESHOLD:", ft["label"], "-", ft["reason"])
        print("   Ship", ft["lastGood"], "or larger.")
    else:
        print("No failure threshold - quality holds across the whole ladder tested.")
    print()
    print("Recommended config by goal:")
    for r in intel.get("recommendations", []):
        rec = r.get("recommendation")
        label = r.get("name") or r.get("profile")
        if rec:
            print("  ", label, "->", rec["quant"], "on", rec["device"])
        else:
            print("  ", label, "-> (no config meets the constraints)")
    print()
    print("Full interactive report:", report.sweep_url)
